# Assignment 4 — Clustering, Web Search, and PageRank

1. **Part 1: Clustering**
   - `readVectorsSeq(filename)`
   - `kcenter(P, k)`
   - `kmeansPP(P, k)`
   - `kmeansObj(P, C)`
   - the required experiments with `k` and `k1`

2. **Part 2: Web Search**
   - inverted index data structures
   - action parser for `actions.txt`
   - verification against `answers.txt`

3. **Part 3: PageRank**
   - iterative PageRank for the graph dataset
   - small graph check
   - final top-5 / bottom-5 reporting



In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Iterable
import csv
import math
import random
import re
import time
import zipfile

import numpy as np

In [16]:
# -------------------------------------------------------------------
ROOT = Path.cwd()

zip_candidates = list(ROOT.glob("Assignment 4- datasets*.zip"))
if zip_candidates and not (ROOT / "Assignment 4- datasets").exists():
    with zipfile.ZipFile(zip_candidates[0]) as zf:
        zf.extractall(ROOT)

DATA_ROOT = ROOT / "Assignment 4- datasets"
Q1_DIR = DATA_ROOT / "Q1- UCI Spam clustering"
Q2_DIR = DATA_ROOT / "Q2- webSearch"
WEB_DIR = Q2_DIR / "webpages"

SPAM_FILE = Q1_DIR / "spambase.data"
ACTIONS_FILE = Q2_DIR / "actions.txt"
ANSWERS_FILE = Q2_DIR / "answers.txt"

# Part 3 graph folder is not included in the zip you uploaded.
# Download the graph folder from the assignment link and place it here:
GRAPH_DIR = ROOT / "Assignment 4- datasets" / "graph"
SMALL_GRAPH_FILE = GRAPH_DIR / "small.txt"
WHOLE_GRAPH_FILE = GRAPH_DIR / "whole.txt"

print("DATA_ROOT:", DATA_ROOT)
print("Spam data exists:", SPAM_FILE.exists())
print("Actions file exists:", ACTIONS_FILE.exists())
print("Answers file exists:", ANSWERS_FILE.exists())
print("Graph folder exists:", GRAPH_DIR.exists())

DATA_ROOT: /home/antpc/DT/Assignment 4- datasets
Spam data exists: True
Actions file exists: True
Answers file exists: True
Graph folder exists: True


## Part 1 — Clustering

In [19]:
try:
    from pyspark.mllib.linalg import Vectors as SparkVectors  # type: ignore
except Exception:
    SparkVectors = None


def make_vector(values: Iterable[float], use_spark_vectors: bool = False):
    values = list(map(float, values))
    if use_spark_vectors and SparkVectors is not None:
        return SparkVectors.dense(values)
    return np.asarray(values, dtype=float)


def as_array(v) -> np.ndarray:
    if hasattr(v, "toArray"):
        return np.asarray(v.toArray(), dtype=float)
    return np.asarray(v, dtype=float)


def sqdist(a, b) -> float:
    da = as_array(a)
    db = as_array(b)
    diff = da - db
    return float(np.dot(diff, diff))

In [6]:
def readVectorsSeq(filename: str, drop_last_column: bool = False, use_spark_vectors: bool = False):
    """
    Read a CSV file where each line is one point.
    Returns a list of vectors.
    """
    path = Path(filename)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {filename}")

    vectors = []
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        reader = csv.reader(f)
        for row in reader:
            if not row:
                continue
            if drop_last_column:
                row = row[:-1]
            vectors.append(make_vector(row, use_spark_vectors=use_spark_vectors))
    return vectors


def kcenter(P, k: int):
    """
    Farthest-first traversal for k-center clustering.
    Time complexity: O(|P| * k)
    """
    if k <= 0:
        raise ValueError("k must be positive")
    if not P:
        return []
    if k >= len(P):
        return list(P)

    centers = [P[0]]  # deterministic start for reproducibility
    min_dists = np.array([sqdist(p, centers[0]) for p in P], dtype=float)

    while len(centers) < k:
        farthest_idx = int(np.argmax(min_dists))
        new_center = P[farthest_idx]
        centers.append(new_center)

        # Update the distance to the nearest chosen center
        for i, p in enumerate(P):
            d = sqdist(p, new_center)
            if d < min_dists[i]:
                min_dists[i] = d

    return centers


def kmeansPP(P, k: int, seed: int = 42):
    """
    k-means++ initialization.
    Time complexity: O(|P| * k)
    """
    if k <= 0:
        raise ValueError("k must be positive")
    if not P:
        return []
    if k >= len(P):
        return list(P)

    rng = random.Random(seed)
    centers = [P[rng.randrange(len(P))]]
    min_dists = np.array([sqdist(p, centers[0]) for p in P], dtype=float)

    while len(centers) < k:
        weights = min_dists.copy()
        total = float(weights.sum())

        if total == 0.0:
            # All remaining points are identical to the chosen centers.
            remaining = [p for p in P if all(sqdist(p, c) != 0 for c in centers)]
            if not remaining:
                break
            next_center = remaining[rng.randrange(len(remaining))]
        else:
            probs = weights / total
            idx = rng.choices(range(len(P)), weights=probs, k=1)[0]
            next_center = P[idx]

        centers.append(next_center)

        for i, p in enumerate(P):
            d = sqdist(p, next_center)
            if d < min_dists[i]:
                min_dists[i] = d

    return centers


def kmeansObj(P, C) -> float:
    """
    Average squared distance from each point in P to its closest center in C.
    """
    if not P:
        raise ValueError("P cannot be empty")
    if not C:
        raise ValueError("C cannot be empty")

    total = 0.0
    for p in P:
        total += min(sqdist(p, c) for c in C)
    return total / len(P)

In [18]:
# -------------------------------------------------------------------
# Load the spam dataset
# -------------------------------------------------------------------
# Set drop_last_column=True if you want to remove the final label column.
# -------------------------------------------------------------------
if SPAM_FILE.exists():
    P = readVectorsSeq(str(SPAM_FILE), drop_last_column=False, use_spark_vectors=False)
    print(f"Loaded {len(P)} points")
    print("Vector length:", len(as_array(P[0])))
else:
    P = []
    print("spambase.data not found. Put the dataset in the Q1 folder first.")

Loaded 4601 points
Vector length: 58


In [17]:
K = 10
K1 = 20

if P:
    # 1) Run kcenter(P, k) and print running time
    t0 = time.perf_counter()
    C_center = kcenter(P, K)
    t1 = time.perf_counter()
    print(f"kcenter(P, {K}) time: {t1 - t0:.6f} seconds")

    # 2) Run kmeansPP(P, k) and then evaluate kmeansObj(P, C)
    C_pp = kmeansPP(P, K, seed=42)
    obj_pp = kmeansObj(P, C_pp)
    print(f"kmeansPP(P, {K}) objective: {obj_pp:.6f}")

    # 3) Run kcenter(P, k1) -> X, then kmeansPP(X, k) -> C, then objective on P
    X = kcenter(P, K1)
    C_coreset = kmeansPP(X, K, seed=42)
    obj_coreset = kmeansObj(P, C_coreset)
    print(f"kcenter(P, {K1}) -> kmeansPP(X, {K}) objective: {obj_coreset:.6f}")

kcenter(P, 10) time: 0.036024 seconds
kmeansPP(P, 10) objective: 31251.603643
kcenter(P, 20) -> kmeansPP(X, 10) objective: 72138.928695


## Part 2 — Web Search (Inverted Index)

In [9]:
STOPWORDS = {"a", "an", "the", "they", "these", "this", "for", "is", "are", "was", "of", "or", "and", "does", "will", "whose"}

# The assignment states that these singular/plural pairs are exhaustive.
# We also include magazine(s) because it appears in the provided sample actions/answers.
PLURAL_TO_SINGULAR = {
    "stacks": "stack",
    "structures": "structure",
    "applications": "application",
    "magazines": "magazine",
}

# Punctuation explicitly listed in the PDF.
PUNCT_RE = re.compile(r"[\{\}\[\]<>=(\)\.,;\'\"\?#!\-:]")

def normalize_word(word: str) -> str:
    w = word.lower()
    if w in PLURAL_TO_SINGULAR:
        return PLURAL_TO_SINGULAR[w]
    # Simple fallback: strip a trailing 's' when it looks like a plural.
    if w.endswith("s") and len(w) > 3 and not w.endswith("ss") and w not in STOPWORDS:
        return w[:-1]
    return w

def tokenize_text(text: str) -> List[str]:
    # Replace punctuation with spaces, then split.
    cleaned = PUNCT_RE.sub(" ", text.lower())
    raw_words = cleaned.split()
    return raw_words

In [10]:
@dataclass(eq=True, frozen=True)
class Position:
    page: "PageEntry"
    wordIndex: int

    def getPageEntry(self):
        return self.page

    def getWordIndex(self):
        return self.wordIndex


class MySet:
    def __init__(self):
        self.elements = []

    def addElement(self, element):
        if element not in self.elements:
            self.elements.append(element)

    def union(self, otherSet: "MySet"):
        out = MySet()
        for e in self.elements:
            out.addElement(e)
        for e in otherSet.elements:
            out.addElement(e)
        return out

    def intersection(self, otherSet: "MySet"):
        out = MySet()
        for e in self.elements:
            if e in otherSet.elements:
                out.addElement(e)
        return out

    def __iter__(self):
        return iter(self.elements)

    def __len__(self):
        return len(self.elements)


class WordEntry:
    def __init__(self, word: str):
        self.word = word.lower()
        self.positions: List[Position] = []

    def addPosition(self, position: Position):
        self.positions.append(position)

    def addPositions(self, positions: List[Position]):
        self.positions.extend(positions)

    def getAllPositionsForThisWord(self):
        return list(self.positions)

    def getTermFrequency(self, word: str):
        # This method is kept for completeness.
        # It returns frequency for the positions stored in this WordEntry.
        if not self.positions:
            return 0.0
        page = self.positions[0].page
        total_words = page.total_words if page.total_words else 1
        return len(self.positions) / total_words


class PageIndex:
    def __init__(self):
        self.word_entries: Dict[str, WordEntry] = {}

    def addPositionForWord(self, str_: str, p: Position):
        if str_ not in self.word_entries:
            self.word_entries[str_] = WordEntry(str_)
        self.word_entries[str_].addPosition(p)

    def getWordEntries(self):
        return list(self.word_entries.values())


class MyHashTable:
    def __init__(self):
        self.table: Dict[str, WordEntry] = {}

    def getHashIndex(self, str_: str):
        # Simple hash index for completeness.
        return hash(str_.lower())

    def addPositionsForWord(self, w: WordEntry):
        key = w.word
        if key not in self.table:
            self.table[key] = w
        else:
            self.table[key].addPositions(w.getAllPositionsForThisWord())

    def getWordEntry(self, str_: str):
        return self.table.get(str_.lower())


class PageEntry:
    def __init__(self, pageName: str, pages_dir: Path = WEB_DIR):
        self.pageName = pageName
        self.pages_dir = Path(pages_dir)
        self.page_path = self.pages_dir / pageName
        if not self.page_path.exists():
            raise FileNotFoundError(f"Webpage file not found: {self.page_path}")

        self.content = self.page_path.read_text(encoding="utf-8", errors="ignore")
        self.pageIndex = PageIndex()
        self.total_words = 0
        self._build_page_index()

    def _build_page_index(self):
        raw_words = tokenize_text(self.content)
        self.total_words = len(raw_words)
        for idx, word in enumerate(raw_words, start=1):
            if word in STOPWORDS:
                continue
            normalized = normalize_word(word)
            self.pageIndex.addPositionForWord(normalized, Position(self, idx))

    def getPageIndex(self):
        return self.pageIndex

    def getHashIndex(self, str_: str):
        return hash(str_.lower())

    def __repr__(self):
        return self.pageName


class InvertedPageIndex:
    def __init__(self):
        self.hashTable = MyHashTable()
        self.pages: Dict[str, PageEntry] = {}

    def addPage(self, p: PageEntry):
        self.pages[p.pageName] = p
        for w in p.getPageIndex().getWordEntries():
            self.hashTable.addPositionsForWord(w)

    def getPagesWhichContainWord(self, str_: str):
        word = normalize_word(str_.lower())
        entry = self.hashTable.getWordEntry(word)
        if entry is None:
            return set()

        pages = set()
        for pos in entry.getAllPositionsForThisWord():
            pages.add(pos.getPageEntry())
        return pages


class SearchEngine:
    def __init__(self, pages_dir: Path = WEB_DIR):
        self.invertedPageIndex = InvertedPageIndex()
        self.pages_dir = Path(pages_dir)

    def performAction(self, actionMessage: str):
        parts = actionMessage.strip().split()
        if not parts:
            return None

        action = parts[0]

        if action == "addPage":
            page_name = parts[1]
            page = PageEntry(page_name, self.pages_dir)
            self.invertedPageIndex.addPage(page)
            return None

        if action == "queryFindPagesWhichContainWord":
            word = parts[1]
            pages = self.invertedPageIndex.getPagesWhichContainWord(word)
            if not pages:
                return f"No webpage contains word {word}"
            return ", ".join(sorted(p.pageName for p in pages))

        if action == "queryFindPositionsOfWordInAPage":
            word, page_name = parts[1], parts[2]
            if page_name not in self.invertedPageIndex.pages:
                return f"No webpage {page_name} found"
            page = self.invertedPageIndex.pages[page_name]
            entry = page.getPageIndex().word_entries.get(normalize_word(word))
            if entry is None:
                return f"Webpage {page_name} does not contain word {word}"
            positions = [p.getWordIndex() for p in entry.getAllPositionsForThisWord()]
            return ", ".join(map(str, positions))

        raise ValueError(f"Unknown action: {action}")

In [11]:
# -------------------------------------------------------------------
# Verify the web-search implementation against answers.txt
# -------------------------------------------------------------------
if ACTIONS_FILE.exists() and ANSWERS_FILE.exists():
    actions = ACTIONS_FILE.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    expected = ANSWERS_FILE.read_text(encoding="utf-8", errors="ignore").strip().splitlines()

    engine = SearchEngine(WEB_DIR)
    got = []
    for action_line in actions:
        result = engine.performAction(action_line)
        if result is not None:
            got.append(result)

    print("Matches answers.txt:", got == expected)
    print()
    for a, g in zip(actions, got):
        print(f"{a} -> {g}")
else:
    print("actions.txt / answers.txt not found. Put the Q2 files in the dataset folder.")

Matches answers.txt: True

addPage stack_datastructure_wiki -> No webpage contains word delhi
queryFindPagesWhichContainWord delhi -> stack_datastructure_wiki
queryFindPagesWhichContainWord stack -> stack_datastructure_wiki
queryFindPagesWhichContainWord wikipedia -> Webpage stack_datastructure_wiki does not contain word magazines
queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki -> No webpage contains word allain
queryFindPagesWhichContainWord allain -> stack_cprogramming
addPage stack_cprogramming -> stack_cprogramming
queryFindPagesWhichContainWord allain -> stack_cprogramming
queryFindPagesWhichContainWord C -> stack_oracle
queryFindPagesWhichContainWord C++ -> stack_cprogramming, stack_datastructure_wiki, stackoverflow
addPage stack_oracle -> stackmagazine


## Part 3 — PageRank on Spark

In [15]:
def parse_edge_line(line: str):
    parts = re.split(r"[,\s]+", line.strip())
    if len(parts) < 2:
        return None
    try:
        u = int(parts[0])
        v = int(parts[1])
        return (u, v)
    except ValueError:
        return None


def read_edges_python(path: Path):
    edges = []
    with Path(path).open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            e = parse_edge_line(line)
            if e is not None:
                edges.append(e)
    # Remove duplicate edges
    return sorted(set(edges))


def pagerank_python(path: Path, beta: float = 0.8, iterations: int = 40):
    edges = read_edges_python(path)
    if not edges:
        raise ValueError(f"No edges found in {path}")

    node_ids = sorted(set(u for u, _ in edges) | set(v for _, v in edges))
    node_to_idx = {node: i for i, node in enumerate(node_ids)}
    idx_to_node = {i: node for node, i in node_to_idx.items()}
    n = len(node_ids)

    out_neighbors = {i: set() for i in range(n)}
    for u, v in edges:
        ui, vi = node_to_idx[u], node_to_idx[v]
        out_neighbors[ui].add(vi)

    outdeg = {i: len(neigh) for i, neigh in out_neighbors.items()}

    ranks = np.full(n, 1.0 / n, dtype=float)

    for _ in range(iterations):
        new_ranks = np.full(n, (1 - beta) / n, dtype=float)
        for i in range(n):
            if outdeg[i] == 0:
                continue
            share = beta * ranks[i] / outdeg[i]
            for j in out_neighbors[i]:
                new_ranks[j] += share
        ranks = new_ranks

    result = [(idx_to_node[i], float(ranks[i])) for i in range(n)]
    result.sort(key=lambda x: x[1], reverse=True)
    return result


def pagerank_spark(path: Path, beta: float = 0.8, iterations: int = 40):
    """
    RDD-based PageRank implementation.
    Requires PySpark.
    """
    from pyspark.sql import SparkSession  # type: ignore

    spark = SparkSession.builder.getOrCreate()
    sc = spark.sparkContext

    edges = (
        sc.textFile(str(path))
        .map(parse_edge_line)
        .filter(lambda x: x is not None)
        .distinct()
        .cache()
    )

    nodes = edges.flatMap(lambda e: [e[0], e[1]]).distinct().collect()
    nodes = sorted(nodes)
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    idx_to_node = {i: node for node, i in node_to_idx.items()}

    edges_idx = (
        edges.map(lambda e: (node_to_idx[e[0]], node_to_idx[e[1]]))
        .distinct()
        .cache()
    )

    outdeg = edges_idx.map(lambda e: (e[0], 1)).reduceByKey(lambda a, b: a + b).collectAsMap()
    outdeg_bc = sc.broadcast(outdeg)

    node_rdd = sc.parallelize(range(n)).cache()
    ranks = node_rdd.map(lambda i: (i, 1.0 / n))

    for _ in range(iterations):
        contribs = (
            edges_idx.join(ranks)  # (src, (dst, rank))
            .map(lambda x: (x[1][0], x[1][1] / outdeg_bc.value[x[0]]))
            .reduceByKey(lambda a, b: a + b)
        )

        ranks = (
            node_rdd.map(lambda i: (i, 0.0))
            .leftOuterJoin(contribs)
            .mapValues(lambda pair: (1 - beta) / n + beta * (pair[1] if pair[1] is not None else 0.0))
        )

    result = ranks.map(lambda x: (idx_to_node[x[0]], x[1])).collect()
    result.sort(key=lambda x: x[1], reverse=True)
    return result


def run_pagerank(path: Path, beta: float = 0.8, iterations: int = 40):
    try:
        import pyspark  # noqa: F401
        return pagerank_spark(path, beta=beta, iterations=iterations)
    except Exception:
        return pagerank_python(path, beta=beta, iterations=iterations)

In [13]:
# -------------------------------------------------------------------
# Sanity checks for PageRank
# -------------------------------------------------------------------
if SMALL_GRAPH_FILE.exists():
    small_scores = run_pagerank(SMALL_GRAPH_FILE, beta=0.8, iterations=40)
    print("Top 5 nodes on small graph:")
    for node, score in small_scores[:5]:
        print(node, score)

    print("\nBottom 5 nodes on small graph:")
    for node, score in small_scores[-5:]:
        print(node, score)
else:
    print("small.txt not found. Place the graph files in the graph/ folder.")

Top 5 nodes on small graph:
53 0.03573120223267159
14 0.03417090697259137
40 0.03363008718974389
1 0.03000597947978861
27 0.029720144201405386

Bottom 5 nodes on small graph:
89 0.003922466019802268
37 0.0038082042916114506
81 0.003695351749360991
59 0.0036698606601272845
85 0.003409694077402821


In [14]:
# -------------------------------------------------------------------
# Final PageRank run on the whole graph
# -------------------------------------------------------------------
if WHOLE_GRAPH_FILE.exists():
    whole_scores = run_pagerank(WHOLE_GRAPH_FILE, beta=0.8, iterations=40)

    top5 = whole_scores[:5]
    bottom5 = sorted(whole_scores, key=lambda x: x[1])[:5]

    print("Top 5 nodes:")
    for node, score in top5:
        print(node, score)

    print("\nBottom 5 nodes:")
    for node, score in bottom5:
        print(node, score)
else:
    print("whole.txt not found. Place the graph files in the graph/ folder.")

Top 5 nodes:
263 0.002020291181518219
537 0.0019433415714531503
965 0.0019254478071662634
243 0.0018526340162417316
285 0.001827372170064515

Bottom 5 nodes:
558 0.0003286018525215297
93 0.00035135689375165774
62 0.00035314810510596274
424 0.0003548153864930145
408 0.00038779848719291705
